# 02 — EDA: Lap Performance

This notebook explores **driver and constructor pace** across the full 2025 F1 season.

| Section | Focus |
|---------|-------|
| 1 | Load & clean raw lap data |
| 2 | Driver lap time distributions |
| 3 | Constructor pace comparison |
| 4 | Sector performance breakdown |
| 5 | Speed trap analysis |
| 6 | Lap time evolution across a race |

> **Prerequisite:** Run `01_data_collection.ipynb` first.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, os.path.join('..', 'src'))
from data_processing import load_lap_data

# ── Style ───────────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Official F1 team colours (2025)
TEAM_COLORS = {
    'McLaren':          '#FF8000',
    'Red Bull Racing':  '#3671C6',
    'Ferrari':          '#E8002D',
    'Mercedes':         '#27F4D2',
    'Aston Martin':     '#229971',
    'Alpine':           '#FF87BC',
    'Williams':         '#64C4FF',
    'RB':               '#6692FF',
    'Kick Sauber':      '#52E252',
    'Haas F1 Team':     '#B6BABD',
}

## 1 · Load & Clean Data

In [ ]:
RAW_PATH = os.path.join('..', 'data', 'raw', 'all_grand_prix_laps_2025.csv')
df = load_lap_data(RAW_PATH)

print(f'Shape : {df.shape}')
print(f'Rounds: {sorted(df["RoundNumber"].dropna().astype(int).unique())}')
print(f'Drivers: {df["Driver"].nunique()}')
df.head(3)

In [ ]:
# Keep only accurate, non-deleted laps with a recorded lap time
laps = df[
    df['IsAccurate'].eq(True) &
    df['Deleted'].eq(False) &
    df['LapTime'].notna()
].copy()

# Remove obvious outliers (SC laps, first-lap chaos, pit-in/out laps)
# Flag lap as a pit lap if PitInTime or PitOutTime is recorded
laps['is_pit_lap'] = laps['PitInTime'].notna() | laps['PitOutTime'].notna()
clean = laps[~laps['is_pit_lap']].copy()

print(f'Accurate non-pit laps: {len(clean):,}  ({len(clean)/len(df)*100:.1f}% of raw)')

## 2 · Driver Lap Time Distributions

Box plots of clean lap times per driver, sorted by median pace.

In [ ]:
# Compute median per driver for sorting
driver_median = (
    clean.groupby('Driver')['LapTime']
    .median()
    .sort_values()
)

fig = px.box(
    clean,
    x='Driver',
    y='LapTime',
    color='Team',
    color_discrete_map=TEAM_COLORS,
    category_orders={'Driver': driver_median.index.tolist()},
    title='Driver Lap Time Distribution — 2025 Season (clean laps)',
    labels={'LapTime': 'Lap Time (s)', 'Driver': ''},
    height=500,
)
fig.update_layout(legend_title='Constructor', xaxis_tickangle=-45)
fig.show()

## 3 · Constructor Pace Comparison

Average lap time per team per race — normalised against the race fastest lap  
so circuits with different base lap times are directly comparable.

In [ ]:
# Fastest lap per race
race_fastest = (
    clean.groupby('EventName')['LapTime']
    .min()
    .rename('race_fastest')
)

team_race = (
    clean.groupby(['EventName', 'RoundNumber', 'Team'])['LapTime']
    .mean()
    .reset_index(name='avg_lap')
    .merge(race_fastest, on='EventName')
)
team_race['gap_pct'] = (team_race['avg_lap'] - team_race['race_fastest']) / team_race['race_fastest'] * 100
team_race.sort_values('RoundNumber', inplace=True)

fig = px.line(
    team_race,
    x='EventName',
    y='gap_pct',
    color='Team',
    color_discrete_map=TEAM_COLORS,
    markers=True,
    title='Constructor Race Pace (% gap to fastest lap) — 2025 Season',
    labels={'gap_pct': '% above fastest lap', 'EventName': ''},
    height=520,
)
fig.update_layout(xaxis_tickangle=-45, legend_title='Constructor')
fig.show()

## 4 · Sector Performance Breakdown

Which sector drives the pace difference between the top teams?

In [ ]:
sector_cols = ['Sector1Time', 'Sector2Time', 'Sector3Time']
sector_data = clean.dropna(subset=sector_cols).copy()

sector_team = (
    sector_data
    .groupby('Team')[sector_cols]
    .mean()
    .reset_index()
    .melt(id_vars='Team', var_name='Sector', value_name='avg_sector_time')
)
sector_team['Sector'] = sector_team['Sector'].str.replace('Time', '')

fig = px.bar(
    sector_team,
    x='Team',
    y='avg_sector_time',
    color='Sector',
    barmode='group',
    title='Average Sector Times by Constructor — 2025 Season',
    labels={'avg_sector_time': 'Avg Time (s)', 'Team': ''},
    height=480,
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 5 · Speed Trap Analysis

Maximum speeds at speed traps reveal straight-line performance and DRS efficiency.

In [ ]:
speed_cols = ['SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST']
speed_data = clean.dropna(subset=speed_cols, how='all').copy()

speed_team = (
    speed_data
    .groupby('Team')[speed_cols]
    .max()
    .reset_index()
    .sort_values('SpeedST', ascending=False)
)

fig = px.bar(
    speed_team,
    x='Team',
    y='SpeedST',
    color='Team',
    color_discrete_map=TEAM_COLORS,
    title='Peak Speed Trap (SpeedST) by Constructor — 2025 Season',
    labels={'SpeedST': 'Max Speed (km/h)', 'Team': ''},
    height=420,
    text='SpeedST',
)
fig.update_traces(texttemplate='%{text:.0f}', textposition='outside')
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

## 6 · Lap Time Evolution During a Race

Select a single race to see how each driver's pace changed lap-by-lap — exposing tyre degradation, fuel effect, and safety car periods.

In [ ]:
# Change this to any EventName in the dataset
SELECTED_RACE = clean['EventName'].iloc[0]
print(f'Showing race: {SELECTED_RACE}')

race = clean[clean['EventName'] == SELECTED_RACE].copy()

fig = px.line(
    race,
    x='LapNumber',
    y='LapTime',
    color='Driver',
    title=f'Lap Time Evolution — {SELECTED_RACE}',
    labels={'LapTime': 'Lap Time (s)', 'LapNumber': 'Lap'},
    height=500,
    markers=True,
)
fig.update_layout(legend_title='Driver')
fig.show()

## 7 · Driver Consistency (Coefficient of Variation)

Consistency = 1 − (std / mean). A value closer to 1 means highly consistent pace.

In [ ]:
consistency = (
    clean.groupby(['Driver', 'Team'])['LapTime']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
consistency.columns = ['Driver', 'Team', 'mean_lap', 'std_lap', 'laps']
consistency = consistency[consistency['laps'] >= 30]  # at least 30 clean laps
consistency['cv_pct'] = consistency['std_lap'] / consistency['mean_lap'] * 100
consistency.sort_values('cv_pct', inplace=True)

fig = px.bar(
    consistency,
    x='Driver',
    y='cv_pct',
    color='Team',
    color_discrete_map=TEAM_COLORS,
    title='Driver Consistency — Coefficient of Variation (lower = more consistent)',
    labels={'cv_pct': 'CV% (std / mean × 100)', 'Driver': ''},
    height=460,
)
fig.update_layout(xaxis_tickangle=-45, showlegend=True)
fig.show()

---
## ✅ Key Takeaways

Fill these in after running:

- **Fastest team overall:** ...
- **Most consistent driver:** ...
- **Sector where the gap is largest:** ...

Proceed to **[03_strategy_analysis.ipynb](03_strategy_analysis.ipynb)**.